# Proposta vs. baselines e vs. variantes de agrupamento

Pergunta: **a proposta (`FuzzyEspecialistEnsembleClassifier`) é melhor que os baselines, e que as variantes de agrupamento, de forma consistente?**

Para cada cenário do `TCC_Experiments`, o experimento foi repetido em vários datasets (Monte Carlo). Cada ponto dos gráficos é a média do CV em **um dataset**.

Dois conjuntos de comparação, cada um com a mesma estrutura:

1. **Proposta vs. baselines** (árvore, Bagging, Boosting, Random Forest).
2. **Proposta vs. variantes de agrupamento** (GMM, FCMdd, PCM, FCM-PFCV), cada uma com a **escolha de c automática**, como a proposta. Só o algoritmo de agrupamento muda; o procedimento de escolha do c é o mesmo. Como o c escolhido também faz parte do efeito, mostro o c médio de cada método.

Para cada conjunto:
- **Painéis por cenário** (3x3): caixas dos modelos, com os pontos de cada dataset ligados entre os modelos (comparação pareada).
- **Mapa de calor** da diferença pareada `proposta - modelo` por cenário, com o número de datasets em que a proposta ganhou e `*` onde o Wilcoxon pareado dá p < 0,05.

Métricas: acurácia e AUC.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

## Configuração

`SCENARIOS` segue o `name` de cada YAML (pasta em `outputs/`). Para cada cenário e modelo uso a execução **mais recente** que contém aquele modelo, então baselines e variantes podem ter sido rodados em execuções diferentes. Para fixar uma execução de um cenário, use `RUNS = {'control': '20261028_153000'}`.

Os dados só se pareiam entre modelos que usaram as mesmas seeds de dataset (`dataset_seed`), ou seja, o mesmo `n_datasets` e a mesma seed base no YAML.

In [ ]:
OUTPUTS = Path('../outputs')
RUNS = {}  # ex.: {'control': '20261028_153000'}

SCENARIOS = {
    'control': 'Controle',
    '1_1_extreme_scale': 'Escala extrema',
    '1_2_feature_imbalance': 'Desbalanceamento de features',
    '2_1_heavy_tail': 'Cauda pesada',
    '2_2_noise_outliers': 'Ruido e outliers',
    '3_1_data_scarcity': 'Escassez de dados',
    '3_2_data_abundance': 'Abundancia de dados',
    '4_1_single_cluster': 'Cluster unico',
    '4_2_fragmentation': 'Fragmentacao',
}

PROPOSAL = 'FuzzyEspecialistEnsembleClassifier'
PROPOSAL_LABEL = 'Proposta'

BASELINES = {
    'DecisionTree': 'Arvore',
    'Bagging': 'Bagging',
    'Boosting': 'Boosting',
    'RandomForestBaseline': 'Random Forest',
}
VARIANTS = {
    'GMMEspecialistEnsembleClassifier': 'GMM',
    'FuzzyKMedoidsEspecialistEnsembleClassifier': 'FCMdd',
    'PossibilisticCMeansEspecialistEnsembleClassifier': 'PCM',
    'FCM_PFCVEspecialistEnsembleClassifier': 'FCM-PFCV',
}

PROPOSAL_COLOR, OTHER_COLOR = '#2a6fdb', '#8a8f98'

In [ ]:
def load_scenario(name):
    """Linhas (dataset x modelo) de um cenario, com a execucao mais recente de cada modelo."""
    folder = OUTPUTS / name
    if not folder.exists():
        return None
    runs = [folder / RUNS[name]] if name in RUNS else sorted(p for p in folder.iterdir() if (p / 'results.csv').exists())
    frames = []
    for run in runs:
        d = pd.read_csv(run / 'results.csv')
        d['run'] = run.name
        frames.append(d)
    if not frames:
        return None
    all_runs = pd.concat(frames, ignore_index=True)
    if 'dataset_idx' not in all_runs:  # CSV de antes das repeticoes Monte Carlo
        all_runs['dataset_idx'] = 0
    if 'dataset_seed' not in all_runs:
        all_runs['dataset_seed'] = all_runs['dataset_idx']
    newest = all_runs.groupby('model')['run'].transform('max')
    out = all_runs[all_runs['run'] == newest].copy()
    out['scenario'] = name
    return out


frames = [f for f in (load_scenario(s) for s in SCENARIOS) if f is not None]
data = pd.concat(frames, ignore_index=True)

LABELS = {PROPOSAL: PROPOSAL_LABEL, **BASELINES, **VARIANTS}
data = data[data['model'].isin(LABELS)].copy()
data['label'] = data['model'].map(LABELS)

# Disponibilidade: nº de datasets por cenario e modelo (vazio = ainda nao rodou)
data.pivot_table(index='scenario', columns='label', values='dataset_idx', aggfunc='count').reindex(
    [s for s in SCENARIOS if s in set(data['scenario'])])

## Funções dos gráficos

In [ ]:
METRICS = {'accuracy_mean': 'Acuracia', 'auc_mean': 'AUC'}


def wide_of(data, scenario, metric, labels):
    """Linhas = datasets (pela seed), colunas = modelos presentes entre `labels`."""
    sub = data[(data['scenario'] == scenario) & data['label'].isin(labels)]
    if sub.empty:
        return None
    wide = sub.pivot(index='dataset_seed', columns='label', values=metric)
    return wide[[l for l in labels if l in wide.columns]]


def grid_boxplots(data, model_set, metric, ncols=3):
    labels = [PROPOSAL_LABEL] + list(model_set.values())
    scenarios = [s for s in SCENARIOS if s in set(data['scenario'])]
    nrows = int(np.ceil(len(scenarios) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 3.8 * nrows), squeeze=False)
    rng = np.random.RandomState(0)

    for ax, scenario in zip(axes.flat, scenarios):
        wide = wide_of(data, scenario, metric, labels)
        if wide is None or PROPOSAL_LABEL not in wide:
            ax.set_visible(False)
            continue
        present = list(wide.columns)
        pos = range(1, len(present) + 1)
        for _, row in wide.iterrows():  # mesmo dataset entre os modelos
            ax.plot(pos, row.values, color='gray', alpha=0.25, lw=0.8, zorder=1)
        bp = ax.boxplot([wide[l].dropna() for l in present], positions=pos, widths=0.5,
                        patch_artist=True, showfliers=False, zorder=2)
        for l, box, med in zip(present, bp['boxes'], bp['medians']):
            color = PROPOSAL_COLOR if l == PROPOSAL_LABEL else OTHER_COLOR
            box.set(facecolor=color, alpha=0.25, edgecolor=color)
            med.set(color=color, lw=2)
        for i, l in zip(pos, present):
            vals = wide[l].dropna()
            ax.scatter(i + rng.uniform(-0.08, 0.08, len(vals)), vals, s=14, zorder=3,
                       color=PROPOSAL_COLOR if l == PROPOSAL_LABEL else OTHER_COLOR)
        ax.set_xticks(list(pos))
        ax.set_xticklabels(present, rotation=30, ha='right')
        ax.set_title(f'{SCENARIOS[scenario]}  (n={len(wide)})')
        ax.grid(alpha=0.25, axis='y')

    for ax in list(axes.flat)[len(scenarios):]:
        ax.set_visible(False)
    fig.suptitle(f'{METRICS[metric]} por cenario (cada ponto = 1 dataset)')
    fig.tight_layout()
    return fig


def diff_heatmap(data, model_set, metric):
    labels = list(model_set.values())
    scenarios = [s for s in SCENARIOS if s in set(data['scenario'])]
    mean = pd.DataFrame(np.nan, index=scenarios, columns=labels)
    text = pd.DataFrame('', index=scenarios, columns=labels)

    for scenario in scenarios:
        wide = wide_of(data, scenario, metric, [PROPOSAL_LABEL] + labels)
        if wide is None or PROPOSAL_LABEL not in wide:
            continue
        for l in labels:
            if l not in wide:
                continue
            d = (wide[PROPOSAL_LABEL] - wide[l]).dropna()
            if d.empty:
                continue
            try:
                star = '*' if wilcoxon(d).pvalue < 0.05 else ''
            except ValueError:  # diferencas todas zero
                star = ''
            mean.loc[scenario, l] = d.mean()
            text.loc[scenario, l] = f'{d.mean():+.3f}{star}\n{(d > 0).sum()}/{len(d)}'

    vmax = np.nanmax(np.abs(mean.values)) if np.isfinite(mean.values).any() else 1
    fig, ax = plt.subplots(figsize=(1.5 * len(labels) + 3.5, 0.6 * len(scenarios) + 1.8))
    im = ax.imshow(mean.values.astype(float), cmap='RdBu', vmin=-vmax, vmax=vmax, aspect='auto')
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels)
    ax.set_yticks(range(len(scenarios)))
    ax.set_yticklabels([SCENARIOS[s] for s in scenarios])
    for i, s in enumerate(scenarios):
        for j, l in enumerate(labels):
            v = mean.loc[s, l]
            dark = np.isfinite(v) and abs(v) > 0.6 * vmax  # texto claro sobre celulas escuras
            ax.text(j, i, text.loc[s, l], ha='center', va='center', fontsize=8,
                    color='white' if dark else 'black')
    fig.colorbar(im, ax=ax, label=f'{METRICS[metric]}: proposta - modelo  (azul = proposta melhor)')
    ax.set_title(f'Diferenca pareada em {METRICS[metric]}  (vitorias/datasets, * = Wilcoxon p<0,05)')
    fig.tight_layout()
    return fig

## 1. Proposta vs. baselines

In [ ]:
grid_boxplots(data, BASELINES, 'accuracy_mean');

In [ ]:
grid_boxplots(data, BASELINES, 'auc_mean');

In [ ]:
diff_heatmap(data, BASELINES, 'accuracy_mean');

In [ ]:
diff_heatmap(data, BASELINES, 'auc_mean');

## 2. Proposta vs. variantes de agrupamento (c automático)

Todos escolhem o c pela mesma votação de métricas de validade. Essas métricas foram pensadas para pertinências do fuzzy c-means, e podem se comportar pior para GMM e PCM. Por isso o c escolhido (tabela abaixo) também vale como resultado.

In [ ]:
grid_boxplots(data, VARIANTS, 'accuracy_mean');

In [ ]:
grid_boxplots(data, VARIANTS, 'auc_mean');

In [ ]:
diff_heatmap(data, VARIANTS, 'accuracy_mean');

In [ ]:
diff_heatmap(data, VARIANTS, 'auc_mean');

In [ ]:
# c medio escolhido por cada metodo (media entre datasets e folds)
labels = [PROPOSAL_LABEL] + list(VARIANTS.values())
chosen_c = data[data['label'].isin(labels)].pivot_table(index='scenario', columns='label', values='c_mean', aggfunc='mean')
chosen_c.reindex([s for s in SCENARIOS if s in chosen_c.index])[[l for l in labels if l in chosen_c.columns]].round(2)

## Notas

(Preencher depois de rodar: em quais cenários a proposta ganha ou perde dos baselines? E das variantes? Os métodos de agrupamento alternativos escolhem um c diferente do FCM?)